# Assignment 3: LLMs and Machine Learning

---

## Submission Instructions

Submit only a link to the folder for Assignment 3 in your personal GitHub repository. Within the repository, you should have a Jupyter notebook file titled e.g. `assignment3.ipynb` or something similar, placed inside the `assignments/assignment3/` folder.

Make sure the repository is public.

**Submissions must be made using a GitHub repository. Submissions that do not follow this instruction will receive 0 points.**

**Late submissions are not accepted as the peer review system does not allow adding submissions past the deadline. Submit your work early to not miss the deadline!**

## Code Quality

Write your code so that it is pleasant to read and easy to understand. This includes:

- Use descriptive variable and function names.
- Add brief comments where the logic is not immediately obvious.
- Keep your notebook organized with clear separation between tasks.
- Print out your answers so that the peer reviewer can see the results. Use the `df.head()` when asked to print the top  5 lines. To print a better looking DataFrame, consider also using `display()` instead of `print()`.
- Divide the code into logical chunks. At minimum, use separate cells per task, and when reasonable, separate cells for subtasks.
- Remember to in the end rerun all code from the beginning to end of the notebook to ensure each cell runs without error

## Visualizations

In the visualizations always include enough information that the plot can be understood independently. This includes:

- Labels for both axes
- A descriptive title

## Statement of use of AI

Include a brief statement describing how and which AI was used (or if no AI was used) in completing the assignment. This could be a markdown cell with a couple of sentences. As a reminder, AI use is permitted in the assignments, but it is advisable to try to complete the tasks as far as possible without and to make sure you understand the code that AI produced when using it.

## Grading

This assignment is worth 10 points. Task 0 is worth 1 point, and tasks 1-2 are worth 2 points and task 3 is worth 5 points.

Points are given only for code that runs. If the code does not run, the task (or subtask if code for a task is divided into multiple cells) will automatically receive 0 points even if the code is almost correct.

### Penalties

- **-2 points per task** where AI-generated (hallucinated) data is used instead of the actual data provided in the task or retrieved from the specified source. The assignment requires working with real data, not made-up values!
- **-3 points** if an API key is included in the submission notebook or anywhere in the GitHub repository. Store your keys in a `.env` file and add `.env` to your `.gitignore`.
- **-1 point** if the Jupyter Notebook is overall messy and not structured well (e.g. if all tasks are completed within one cell, if answers are difficult to find due to too much irrelevant printed output).
- **-1 point** if there is no statement of AI use. If you did not use AI, report that you did not use AI.

### Editing the submission after the deadline

- Editing the assignment submission during the evaluation phase is forbidden. Thus, after the solution has been released, do not make any further changes to the notebook until you have received a grade. If you accidentally leaked an API key, revoke the key immediately. Other **changes to the submission are considered cheating, and will result in 0 points for both the assignment and peer review**.

---

## Tasks

### Task 0: Setting up Ollama (1p)

a) Set up Ollama and connect to it using either openAI's API or Ollama's own API. 

b) Load the 270m parameter version of the [gemma3](https://ollama.com/library/gemma3) model and test it with any prompt.

c) Load the 4b parameter version of the [gemma3](https://ollama.com/library/gemma3) and test it with any prompt. If running the 4b version is too slow, you can use the 1b version instead.

In [19]:
# a)
# Following the example from L5_3 after downloading the reqyusute gemma3 models
# If needed, the openai install needs to be run
#!pip install openai

from openai import OpenAI

model_270m = "gemma3:270m"
model_4b = "gemma3:4b"

# Connecting to Ollama using the openAI API
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama"  # required by the library but ignored by Ollama
)
print("Ollama connection initialized.")

Ollama connection initialized.


In [20]:
# b)
# Now, load each of the two downloaded models and test connecting to them using the example code
# from L5_3. Extending the example function functionality by adding a second input parameter 'model'
# Starting with the 270m model.

def ask_llm(prompt, model):
    """Send a prompt to the local LLM and return the response text."""
    response = ollama_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [21]:
print(ask_llm("What is sentiment analysis? Answer in one sentence.", model_270m))

Sentiment analysis is the process of identifying and analyzing the emotional tone or attitude expressed in text data, typically using machine learning models to extract meaning and relationships between words and concepts.



In [22]:
# c)
# Loading and testing the 4b parameter model. Again, using the example code from L5_3 and
# the ask_llm -function created previously. Now, piping model_4b to the function.

print(ask_llm("What is sentiment analysis? Answer in one sentence.", model_4b))

Sentiment analysis is the process of automatically determining the emotional tone or subjective attitude expressed in a piece of text, such as a review, social media post, or article.


### Task 1: Text classification with Ollama (2p)

The `data/emails.csv` file contains 12 email headlines, with 4 spam emails, 4 legitimate work emails and 4 vague emails that are hard to classify based on the title alone. Use this dataset for all subtasks in this task.

a) Make a function for classifying emails (based on the headlines) as spam, work or unknown. The function should return only the classification and nothing else. (0.5p)

b) Use the smaller gemma3 (270m) to classify the emails using the function created in part a. (0.5p)

c) Use larger gemma3 (4b) to classify the emails using the function created in part a). In separate markdown cell, write a brief comment comparing the results of parts b) and c). (0.5p)

d) Write a script that repeats b) and c) 3 times, storing the results for both models separately. For both models, put the results as columns into a new DataFrame that also contains the headlines so that it is easy to compare how the output varied across runs for both models. Comment if there were differences and explain why this happened. (0.5p)

In [23]:
# Starting by loading the data and checking its contents
import pandas as pd

# Reading the csv initially fails due to the additional commas in the headline strings
# Fixed this by setting the csv delimiter to ";" in the csv-file (within JupyterLab).
df_emails = pd.read_csv("data/emails.csv", sep=";")
df_emails

,headline
0,URGENT: Your account will be suspended within ...
1,Congratulations! You have won a 1000€ gift car...
2,Hot singles in your area are waiting to meet y...
3,Re: Inheritance transfer of 4.5M USD pending y...
4,Meeting agenda for Thursday's project review
5,"Q3 budget report attached, please review by Fr..."
6,Reminder: Annual performance review scheduled ...
7,"Updated draft of the manuscript, comments welcome"
8,Quick question about last week
9,Following up


In [28]:
# a)
# Constructing a function based on Task 3.2 in the text analysis notebook of L5_3

CATEGORIES = ["Spam", "Work", "Unknown"]

def classify_email(text, model):
    """Classify an email headline into one of the predefined categories."""
    categories_str = ", ".join(CATEGORIES)
    prompt = f"""Classify the following email headline into exactly one of these categories: {categories_str}.
Respond with only the category name, nothing else.

Headline: {text}"""
    result = ask_llm(prompt, model)
    return result.strip()

# Test on the first email headline
print(df_emails.loc[0, "headline"])
print()
print(classify_email(df_emails.loc[0, "headline"], model_4b))

URGENT: Your account will be suspended within 24 hours

Spam


In [33]:
# b)
# Starting off by making a copy of the DataFrame and adding the results from subtasks b) and c).
# This will allow for easy printing of the df
df_emails_bc = df_emails.copy()

# Now we can classify the emails and add the results to columns of the copied df
# Creating a new column name "270m" and piping the classify_email function call using a temporary
# lambda function, similarly to L5_3 notebook task 3.3
df_emails_bc["270m"] = df_emails_bc["headline"].apply(lambda x: classify_email(x, model_270m))
df_emails_bc

,headline,270m
0,URGENT: Your account will be suspended within ...,Spam
1,Congratulations! You have won a 1000€ gift car...,Work
2,Hot singles in your area are waiting to meet y...,Work
3,Re: Inheritance transfer of 4.5M USD pending y...,Spam
4,Meeting agenda for Thursday's project review,Work
5,"Q3 budget report attached, please review by Fr...",Email
6,Reminder: Annual performance review scheduled ...,Work
7,"Updated draft of the manuscript, comments welcome",Work
8,Quick question about last week,Spam
9,Following up,Spam


In [34]:
# c)
# Classification using the 4b model following the same code as b), with the model changing
df_emails_bc["4b"] = df_emails_bc["headline"].apply(lambda x: classify_email(x, model_4b))
df_emails_bc

,headline,270m,4b
0,URGENT: Your account will be suspended within ...,Spam,Spam
1,Congratulations! You have won a 1000€ gift car...,Work,Spam
2,Hot singles in your area are waiting to meet y...,Work,Spam
3,Re: Inheritance transfer of 4.5M USD pending y...,Spam,Spam
4,Meeting agenda for Thursday's project review,Work,Work
5,"Q3 budget report attached, please review by Fr...",Email,Work
6,Reminder: Annual performance review scheduled ...,Work,Work
7,"Updated draft of the manuscript, comments welcome",Work,Work
8,Quick question about last week,Spam,Unknown
9,Following up,Spam,Unknown


#### Brief comparison of the results from b) and c)
Both models return one word classifications. But the 270m model, while faster, does not adhere to the strict classification label guidelines. This smaller model does not seem to grasp the ambiguity of the "Unknown" classification, preferring to classify all email headlines as either "Work" or "Spam". The 4b model seems much more accurate in its classifications and can assess the ambiguity of the headlines in 8, 9 and 11. 

In [42]:
# d)
# Creating another copy of the original DataFrame and adding the iterative results to this df
df_emails_classified = df_emails.copy()

# Creating a list of the models to iterate through
models = [model_270m, model_4b]

# This script will iterate through each model and for each, further iterate 3 times,
# applying the same logic used in b) and c)
for model in models:
    for i in range(1, 4):
        column_name = f"{model}_run_{i}"
        print(f"Executing {column_name}...")
        df_emails_classified[column_name] = df_emails_classified["headline"].apply(lambda x: classify_email(x, model))
print("The classification script has executed.")

Executing gemma3:270m_run_1...
Executing gemma3:270m_run_2...
Executing gemma3:270m_run_3...
Executing gemma3:4b_run_1...
Executing gemma3:4b_run_2...
Executing gemma3:4b_run_3...
The classification script has executed.


In [44]:
# Printing the full results for comparison
df_emails_classified

,headline,gemma3:270m_run_1,gemma3:270m_run_2,gemma3:270m_run_3,gemma3:4b_run_1,gemma3:4b_run_2,gemma3:4b_run_3
0,URGENT: Your account will be suspended within ...,Spam,Spam,Spam,Spam,Spam,Spam
1,Congratulations! You have won a 1000€ gift car...,Spam,Spam,Work,Spam,Spam,Spam
2,Hot singles in your area are waiting to meet y...,Spam,Spam,Work,Spam,Spam,Spam
3,Re: Inheritance transfer of 4.5M USD pending y...,Work,Work,Spam,Spam,Spam,Spam
4,Meeting agenda for Thursday's project review,Work,Work,Work,Work,Work,Work
5,"Q3 budget report attached, please review by Fr...",Work,Work,Work,Work,Work,Work
6,Reminder: Annual performance review scheduled ...,Work,Work,Work,Work,Work,Work
7,"Updated draft of the manuscript, comments welcome",Work,Work,Work,Work,Work,Work
8,Quick question about last week,Spam,Spam,Spam,Unknown,Unknown,Unknown
9,Following up,Spam,Spam,Spam,Unknown,Unknown,Unknown


#### Quick comparison of the iterative results in d)
Iterating through the classification process multiple times highlights the inherent variable results of the smaller 270m model while showcasing the stability of results of the larger 4m model. Both LLM models have a degree of randomness, but the larger 4m model still proves more decisive in its classification. The smaller 270m model exhibits higher output variance and tends to not adhere to the strictly defined prompt instructions as well as the larger model. As a result, the 4m model output is consistent while the smaller model output exhibits much more run-to-run variablity. 

### Task 2: Sentiment analysis with Ollama (2p)

The `data/news.csv` file contains 10 fictional financial news headlines. Use it for all subtasks in this task.

a) Make a function for classifying the texts in the provided dataset based on the topic (earnings, mergers, regulation, macroeconomics) and for determining the sentiment of the news (positive, negative, neutral). The function should return the class and sentiment in JSON format. (1p)

b) Use gemma3 (4b) to classify and provide the sentiment for each row of the provided dataset, inserting them into a new DataFrame that contains both the original headlines as well as topic and sentiment. (0.5p)

c) Give the same data and prompt to a browser based LLM (e.g. ChatGPT, LeChat, Claude or Gemini) and ask it to provide the topic and sentiment, giving it the same options. Paste the results into a markdown cell. Compare the results of b) and c), which one is more accurate and why? (0.5p)

In [46]:
# Starting by reading the data file and checking its content
# Again, reading the csv runs into errors due to the presence of commas in text strings
# Setting the delimiter of the csv-file to ";" can solve this issue
df_news = pd.read_csv("data/news.csv", sep=";")
df_news

,headline
0,Nordion Industries beats Q1 earnings estimates...
1,Helvora Pharmaceuticals misses earnings foreca...
2,"Aurelis Bank reports steady quarterly profit, ..."
3,Veridyne Logistics to acquire rival Trantec in...
4,Antitrust regulators block proposed merger bet...
5,Kestrel Semiconductor confirms early-stage mer...
6,New EU AI Act compliance rules expected to rai...
7,Finnish FSA grants Norvik Capital expanded lic...
8,"Eurozone inflation cools to 2.1%, easing press..."
9,Rising interest rates weigh on Tessaro Real Es...


In [50]:
# a)
# Creating a function based on the examples from L5_3
import json

def news_analysis(headline, model):
    """Classify a headline by topic and sentiment, returning a JSON object."""
    categories_str = ", ".join(["Earnings", "Mergers", "Regulation", "Macroeconomics"])
    prompt = f"""Classify the following financial headline.
Return your answer as a JSON object with two keys:
- "category": one of {categories_str}
- "sentiment": one of "Positive", "Negative", "Neutral"

Do not include any other text, markdown formatting, or code blocks. Return only the JSON object.

Message: {headline}"""
    result = ask_llm(prompt, model)
    cleaned = result.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(cleaned)

# Test on the billing ticket
print(df_news.loc[0, "headline"])
detail = news_analysis(df_news.loc[0, "headline"], model_4b)
print(json.dumps(detail, indent=2))

Nordion Industries beats Q1 earnings estimates as cloud revenue surges 28%
{
  "category": "Earnings",
  "sentiment": "Positive"
}


In [53]:
# b)
# Making a copy of the df and adding the analysis
df_news_analyzed = df_news.copy()

# Classification and sentiment analysis of each headline
details = df_news_analyzed["headline"].apply(lambda x: news_analysis(x, model_4b))

# Parsing the resulting JSON to the DataFrame
df_news_analyzed["category"] = details.apply(lambda x: x.get("category", ""))
df_news_analyzed["sentiment"] = details.apply(lambda x: x.get("sentiment", ""))

# Printing to check the script worked as intended
df_news_analyzed

,headline,category,sentiment
0,Nordion Industries beats Q1 earnings estimates...,Earnings,Positive
1,Helvora Pharmaceuticals misses earnings foreca...,Earnings,Negative
2,"Aurelis Bank reports steady quarterly profit, ...",Earnings,Positive
3,Veridyne Logistics to acquire rival Trantec in...,Mergers,Neutral
4,Antitrust regulators block proposed merger bet...,Regulation,Negative
5,Kestrel Semiconductor confirms early-stage mer...,Mergers,Neutral
6,New EU AI Act compliance rules expected to rai...,Regulation,Negative
7,Finnish FSA grants Norvik Capital expanded lic...,Regulation,Positive
8,"Eurozone inflation cools to 2.1%, easing press...",Macroeconomics,Neutral
9,Rising interest rates weigh on Tessaro Real Es...,Macroeconomics,Negative


#### c) Browser-based LLM analysis results (Gemini 3.1 Pro)
|Headline|Topic|Sentiment|
|---------|---------|-------|
Nordion Industries beats Q1 earnings estimates as cloud revenue surges 28%|earnings|positive
Helvora Pharmaceuticals misses earnings forecast amid weak generics demand|earnings|negative
"Aurelis Bank reports steady quarterly profit, in line with analyst expectations"|earnings|neutral
Veridyne Logistics to acquire rival Trantec in 4.2 billion euro deal|mergers|neutral
Antitrust regulators block proposed merger between Solenta and Marvex Energy|regulation|negative
Kestrel Semiconductor confirms early-stage merger talks with Aldenfeld AG|mergers|positive
New EU AI Act compliance rules expected to raise costs for Lumavex by 12%|regulation|negative
Finnish FSA grants Norvik Capital expanded licence for cross-border operations|regulation|positive
"Eurozone inflation cools to 2.1%, easing pressure on Drava Holdings borrowing costs"|macroeconomics|positive
Rising interest rates weigh on Tessaro Real Estate as financing costs climb|macroeconomics|negative


#### Comparison of the results from b) and c)
Both the local 4b model and the browser-based model classify the headlines identically under the 4 given topics (earnings, mergers, regulation, macroeconomics). Therefore, the comparatively small local 4b parameter model performs admirably when categorizing the headline information. On the other hand, the models differ quite significantly in their sentiment analysis. From this run of the analysis, the sentiment analysis differs here:

|Headline|4b|Gemini|
|-----------|------|------|
Aurelis Bank reports steady quarterly profit, in line with analyst expectations|positive|neutral
Kestrel Semiconductor confirms early-stage merger talks with Aldenfeld AG|neutral|positive
Eurozone inflation cools to 2.1%, easing pressure on Drava Holdings borrowing costs|neutral|positive

The sentiment analysis requires a bit of nuanced reading, while not always even exhibiting a clear-cut correct answer. The larger Gemini model (often hundreds of millions of parameters, exact number not disclosed for this 3.1 Pro model) can likely identify contextual nuances way better than the 4b local Ollama model. This often provides a bit more accurate sentiment analysis. Note that the differences between the models is limited to neutral/positive, with no wild discrepencies in the sentiment in terms of positive/negative. The smaller 4b model might tend towards a conservative evaluation when the headline context becomes murky, while the larger model can eke out a more exact reading. While both models work quite well, with no visible outlier results or clear hallucination present, the larger browser-based model tends to be a bit smarter in its sentiment assessment, thus more accurate.

### Task 3: Supervised machine learning (5p)

For this task, use a subset of the [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) dataset, by downloading and importing the `bank-additional.csv` from the UCI repository. You can find a description of the dataset behind the link.

The goal is to predict whether a prospective customer will subscribe to a term deposit (variable y).

a) Import the dataset and conduct exploratory data analysis on it. (1p)

b) Preprocess the data using the appropriate methods as described in the course materials. (1p)

c) Determine whether this is a classification or regression task. Choose three different machine learning algorithms from scikit-learn and explain briefly why you chose them. For each of the selected algorithsm, train and a model and iteratively adjust the hyperparameters until you no longer manage to improve the performance. (1p)

d) Compare using train, validation and test set split versus using cross-validation. Which one performs better? (1p)

e) Report and evaluate the performance of the models using several of the metrics provided in the course, and explain which model is the best for the task and why. (1p)
